In [1]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import random

In [ ]:
from sklearn.svm import SVR
import pandas as pd
from collections import defaultdict

In [ ]:
def dynaq(env, alpha=0.1, gamma=0.99, epsilon=0.1, episodes=1000, planning=10):

    #Initialization
    n_states = env.observation_space.n
    n_current_actions = env.current_action_space.n
    Q = np.zeros((n_states, n_current_actions))

    model = {}

    for episode in range(episodes): #simuliamo più volte l'environment

        done = False

        #Epsilon decay
        epsilon = max(0.01, epsilon - 0.01)
        state = env.reset()[0]  # Reset environment to initial state

        while not done: #itera tra tutte le azioni prese nell'episodio

            # Epsilon-greedy current_action selection
            if random.uniform(0, 1) < epsilon:
                current_action = env.current_action_space.sample()
            else:
                current_action = np.argmax(Q[state, :])

            #Taking current_action
            next_state, reward, done, _, _ = env.step(current_action)

            model[(state, current_action)] = (reward, next_state)

            #Predicting next_current_action
            if random.uniform(0, 1) < epsilon:
                next_action = env.current_action_space.sample()
            else:
                next_action = np.argmax(Q[next_state, :])

            #Updating Q[state, current_action]
            td_target = reward + gamma * Q[next_state, next_action]
            td_error = td_target - Q[state, current_action]
            Q[state, current_action] += alpha * td_error

            for _ in range(planning):

                if not model:
                    break

                s, a = random.choice(list(model.keys()))
                r, s_prime = model[(s, a)]

                a_prime = np.argmax(Q[s_prime, :])

                td_target = r + gamma * Q[s_prime, a_prime]
                td_error = td_target - Q[s, a]

                Q[s, a] += alpha * td_error

            state = next_state
            
    policy = np.argmax(Q, axis=1)

    return Q, model, policy